# Customer Churn Prediction
## A Data Science Project for Udemy Certification

**Author:** [Your Name]  
**Date:** July 2026  
**Course:** Data Science Certification - Udemy

---

### Business Problem

Customer churn is one of the biggest challenges for subscription-based businesses. A telecom company wants to identify customers who are likely to churn so they can take proactive retention measures. This project builds a machine learning pipeline to predict churn probability.

### Objectives
1. Perform Exploratory Data Analysis (EDA) to understand churn patterns
2. Engineer features and preprocess data for modeling
3. Train and compare multiple ML models (Logistic Regression, Random Forest, XGBoost)
4. Evaluate models using appropriate metrics and select the best performer
5. Derive actionable business insights

### Key Metrics
- **Recall (Sensitivity):** How many actual churners do we correctly identify?
- **Precision:** Of those we flag as churners, how many actually churn?
- **F1-Score:** Harmonic mean of precision and recall (balanced metric)
- **ROC-AUC:** Overall discriminatory power of the model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("Set2")
%matplotlib inline

print("Libraries loaded successfully!")

---
## 1. Data Loading and Initial Inspection

In [ ]:
df = pd.read_csv("../data/telco_churn.csv")
print(f"Dataset Shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
print("Data Info:")
df.info()

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
print("Target Distribution:")
churn_counts = df["Churn"].value_counts()
print(f"No Churn (0): {churn_counts.get(0, 0)} ({churn_counts.get(0, 0)/len(df)*100:.1f}%)")
print(f"Churn (1):    {churn_counts.get(1, 0)} ({churn_counts.get(1, 0)/len(df)*100:.1f}%)")

---
## 2. Exploratory Data Analysis (EDA)

### 2.1 Target Variable Visualization

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#2ecc71", "#e74c3c"]
ax[0].pie(
    churn_counts.values,
    labels=["No Churn", "Churn"],
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
    explode=(0, 0.05),
)
ax[0].set_title("Churn Distribution", fontsize=14, fontweight="bold")

bars = ax[1].bar(["No Churn", "Churn"], churn_counts.values, color=colors, edgecolor="white")
for bar, val in zip(bars, churn_counts.values):
    ax[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, str(val), ha="center", fontweight="bold")
ax[1].set_title("Churn Count", fontsize=14, fontweight="bold")
ax[1].set_ylabel("Number of Customers")

plt.tight_layout()
plt.savefig("../reports/churn_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.2 Demographic Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

demographic_cols = ["gender", "SeniorCitizen", "Partner", "Dependents"]
titles = ["Gender", "Senior Citizen", "Has Partner", "Has Dependents"]

for ax, col, title in zip(axes.flat, demographic_cols, titles):
    churn_rate = df.groupby(col)["Churn"].mean() * 100
    bars = ax.bar(
        ["Female" if x == "Female" or x == 1 else "Male" if x == "Male" else "No" if x == "No" or x == 0 else "Yes"
         for x in churn_rate.index],
        churn_rate.values,
        color=["#3498db", "#e74c3c"],
        edgecolor="white",
    )
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{val:.1f}%", ha="center", fontweight="bold")
    ax.set_title(f"Churn Rate by {title}", fontsize=13, fontweight="bold")
    ax.set_ylabel("Churn Rate (%)")
    ax.set_ylim(0, max(churn_rate.values) * 1.3)

plt.tight_layout()
plt.savefig("../reports/demographic_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** Senior citizens and customers without partners/dependents show higher churn rates.

### 2.3 Service & Contract Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

service_cols = ["InternetService", "Contract", "PaymentMethod", "PaperlessBilling"]
titles = ["Internet Service", "Contract Type", "Payment Method", "Paperless Billing"]

for ax, col, title in zip(axes.flat, service_cols, titles):
    churn_rate = df.groupby(col)["Churn"].mean().sort_values(ascending=False) * 100
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(churn_rate)))
    bars = ax.barh(range(len(churn_rate)), churn_rate.values, color=colors, edgecolor="white")
    ax.set_yticks(range(len(churn_rate)))
    ax.set_yticklabels(churn_rate.index, fontsize=9)
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f"{val:.1f}%", va="center", fontweight="bold")
    ax.set_title(f"Churn Rate by {title}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Churn Rate (%)")
    ax.set_xlim(0, max(churn_rate.values) * 1.3)

plt.tight_layout()
plt.savefig("../reports/service_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** Month-to-month contracts and fiber optic users churn significantly more. Electronic check payers also show elevated churn.

### 2.4 Numerical Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
titles = ["Tenure (Months)", "Monthly Charges ($)", "Total Charges ($)"]

for ax, col, title in zip(axes, numeric_cols, titles):
    for churn_val, label, color in [(0, "No Churn", "#2ecc71"), (1, "Churn", "#e74c3c")]:
        subset = df[df["Churn"] == churn_val][col]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor="white")
    ax.set_title(f"Distribution of {title}", fontsize=13, fontweight="bold")
    ax.set_xlabel(title)
    ax.set_ylabel("Count")
    ax.legend()

plt.tight_layout()
plt.savefig("../reports/numerical_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** Churners tend to have lower tenure and higher monthly charges.

### 2.5 Correlation Heatmap

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.drop("customerID", axis=1).copy()
le = LabelEncoder()
for col in df_encoded.select_dtypes(include="object").columns:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

plt.figure(figsize=(16, 12))
corr = df_encoded.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)
plt.title("Feature Correlation Matrix", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3. Data Preprocessing & Feature Engineering

In [ ]:
import sys
sys.path.append("../src")
from preprocessing import load_data, clean_data, encode_categorical, prepare_data

X_train, X_test, y_train, y_test, scaler, feature_names = prepare_data(
    "../data/telco_churn.csv", test_size=0.2, random_state=42
)

print(f"Training set:   {X_train.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")
print(f"Features:       {X_train.shape[1]}")
print(f"Churn (Train):  {y_train.mean():.2%}")
print(f"Churn (Test):   {y_test.mean():.2%}")
print(f"\nClass distribution:")
print(f"  Train - No Churn: {(y_train == 0).sum()}, Churn: {(y_train == 1).sum()}")
print(f"  Test  - No Churn: {(y_test == 0).sum()}, Churn: {(y_test == 1).sum()}")

Since the dataset is imbalanced (more non-churners than churners), we use **SMOTE (Synthetic Minority Oversampling Technique)** to balance the training data.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {X_train.shape[0]} samples")
print(f"After SMOTE:  {X_train_res.shape[0]} samples")
print(f"New class distribution: No Churn={(y_train_res==0).sum()}, Churn={(y_train_res==1).sum()}")

---
## 4. Model Training & Evaluation

We train three models and compare them:

1. **Logistic Regression** — Simple, interpretable baseline
2. **Random Forest** — Ensemble method, handles non-linearity well
3. **XGBoost** — Gradient boosting, state-of-the-art for tabular data

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=10,
        min_samples_leaf=4, random_state=42, class_weight="balanced", n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_train_res==0).sum()/(y_train_res==1).sum(),
        random_state=42, use_label_encoder=False, eval_metric="logloss"
    ),
}

results = {}
probabilities = {}

for name, model in models.items():
    print(f"Training {name}...", end=" ")
    model.fit(X_train_res, y_train_res)
    print("Done!")

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    }
    probabilities[name] = y_proba

print("\nAll models trained successfully!")

### 4.1 Model Comparison

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df.columns))
width = 0.25

colors = ["#3498db", "#2ecc71", "#e74c3c"]
for i, (model_name, row) in enumerate(results_df.iterrows()):
    bars = ax.bar(x + i * width, row.values, width, label=model_name, color=colors[i], edgecolor="white")
    for bar, val in zip(bars, row.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.3f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xlabel("Metric", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Model Performance Comparison", fontsize=15, fontweight="bold")
ax.set_xticks(x + width)
ax.set_xticklabels(results_df.columns, fontsize=11)
ax.legend(loc="lower right", fontsize=11)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig("../reports/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nDetailed Results:")
display(results_df.style.background_gradient(cmap="RdYlGn", axis=1).format("{:.4f}"))

### 4.2 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for name, color in zip(probabilities.keys(), ["#3498db", "#2ecc71", "#e74c3c"]):
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    auc = roc_auc_score(y_test, probabilities[name])
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=color, linewidth=2.5)

ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves - Model Comparison", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=ax,
        xticklabels=["No Churn", "Churn"],
        yticklabels=["No Churn", "Churn"],
        annot_kws={"fontsize": 14, "fontweight": "bold"},
    )
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig("../reports/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.4 Detailed Classification Reports

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

---
## 5. Feature Importance Analysis

In [ ]:
best_model_name = results_df["F1-Score"].idxmax()
best_model = models[best_model_name]

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
elif hasattr(best_model, "coef_"):
    importances = np.abs(best_model.coef_[0])
else:
    importances = None

if importances is not None:
    feature_cols = X_train.columns
    feat_importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": importances,
    }).sort_values("Importance", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(feat_importance_df)))
    ax.barh(feat_importance_df["Feature"], feat_importance_df["Importance"], color=colors, edgecolor="white")
    ax.set_xlabel("Importance", fontsize=12)
    ax.set_title(f"Feature Importance ({best_model_name})", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../reports/feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    print("\nTop 10 Most Important Features:")
    display(feat_importance_df.tail(10)[::-1])

---
## 6. Business Recommendations

Based on our analysis, here are actionable recommendations:

### 1. Target Month-to-Month Contract Holders
- Offer discounts for switching to annual contracts
- Run loyalty programs specifically for short-term contract customers

### 2. Address Fiber Optic Churn
- Investigate service quality issues with fiber optic customers
- Bundle premium support services with fiber optic plans

### 3. Encourage Auto-Payment
- Electronic check users churn significantly more
- Offer small discounts for switching to credit card or bank transfer auto-pay

### 4. Upsell Value-Added Services
- Online security and tech support correlate with lower churn
- Bundle these services at a modest price increase

### 5. Early Intervention
- Deploy the predictive model in production
- Flag high-risk customers early and trigger retention workflows

---

## 7. Conclusion

This project demonstrates a complete data science workflow:

- **EDA** revealed key churn drivers: contract type, tenure, payment method
- **Data preprocessing** handled missing values, encoding, scaling, and class imbalance via SMOTE
- **Three ML models** were trained and compared comprehensively
- The **best model** (XGBoost/Random Forest) achieved strong performance across all metrics
- **Business recommendations** were derived directly from the analysis

The model can be deployed in production to score customers and trigger retention actions, potentially saving the business significant revenue.